# 📈 Regression Analysis — D2–D5

Phân tích mối liên hệ giữa kích thước code, thời gian đến lần thay đổi đầu tiên và các đặc trưng của khối code có comment AI. Đây là phân tích liên hệ, không tự chứng minh quan hệ nhân quả.

- Mỗi dòng là một khối code duy nhất. Sai số chuẩn tính theo cụm repo.
- Đã bỏ D1 (`code_lines ~ days_to_modification`): R² = 0,002 và mô hình không có cách diễn giải rõ ràng.
- Đã bỏ `task_type` dạng regex vì nhãn không chính xác.

Cần cài thêm: `statsmodels`, `patsy`.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
DATA_PATH = Path('../data/processed/commits_features.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
df['days_to_modification'] = pd.to_numeric(df['days_to_modification'], errors='coerce')
df['has_recorded_change'] = df['has_recorded_change'].astype(int)
df = df.replace([np.inf, -np.inf], np.nan)
REPORT_DIR = Path('../reports'); FIG_DIR = REPORT_DIR / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
def cluster(d):
    # sai số chuẩn theo cụm repo vì các khối cùng repo không độc lập
    return {'cov_type': 'cluster', 'cov_kwds': {'groups': pd.factorize(d['repo'])[0]}}
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_DIR / name, dpi=160, bbox_inches='tight'); plt.show(); plt.close()
print(f'Đã tải {len(df):,} dòng × {df.shape[1]} cột')

Đã tải 21,498 dòng × 23 cột


## D2 — Hồi quy tuyến tính bội
`days_to_modification ~ code_lines + repo_commit_count + language + ai_tool`

Chỉ dùng khối có khoảng thời gian > 0. `commit_date` là cận trên của ngày đưa code vào, nên D2 là kết quả phụ.

In [2]:
d2_cols = ['days_to_modification','code_lines','repo_commit_count','language','ai_tool','repo']
d2 = df[d2_cols].dropna().copy()
d2_model = smf.ols('days_to_modification ~ code_lines + repo_commit_count + C(language) + C(ai_tool)', data=d2).fit(**cluster(d2))
print(d2_model.summary())
d2_result = pd.DataFrame({'term': d2_model.params.index, 'coef': d2_model.params.values, 'p_value': d2_model.pvalues.values, 'ci_lower': d2_model.conf_int()[0].values, 'ci_upper': d2_model.conf_int()[1].values})
d2_result.to_csv(REPORT_DIR / 'regression_d2_multiple.csv', index=False)
plt.figure(figsize=(9, max(5, len(d2_result)*.25))); coef = d2_result[d2_result.term != 'Intercept'].sort_values('coef'); plt.errorbar(coef.coef, range(len(coef)), xerr=[coef.coef-coef.ci_lower, coef.ci_upper-coef.coef], fmt='o'); plt.yticks(range(len(coef)), coef.term); plt.axvline(0,color='black',lw=1); plt.title('D2: coefficient plot'); plt.xlabel('Hệ số (95% CI)'); savefig('D2_coefficient_plot.png')

                             OLS Regression Results                             
Dep. Variable:     days_to_modification   R-squared:                       0.025
Model:                              OLS   Adj. R-squared:                  0.024
Method:                   Least Squares   F-statistic:                     612.4
Date:                  Thu, 12 Nov 2026   Prob (F-statistic):               0.00
Time:                          10:02:11   Log-Likelihood:                -32689.
No. Observations:                  5758   AIC:                         6.540e+04
Df Residuals:                      5748   BIC:                         6.546e+04
Df Model:                             9                                         
Covariance Type:                cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------

## D3 — Hồi quy logistic
`has_recorded_change ~ code_lines + repo_commit_count + language + ai_tool + year`

In [3]:
d3_cols = ['has_recorded_change','code_lines','repo_commit_count','language','ai_tool','year','repo']
d3 = df[d3_cols].dropna().copy()
d3_model = smf.glm('has_recorded_change ~ code_lines + repo_commit_count + C(language) + C(ai_tool) + year', data=d3, family=sm.families.Binomial()).fit(**cluster(d3))
print(d3_model.summary())
d3_result = pd.DataFrame({'term': d3_model.params.index, 'log_odds_coef': d3_model.params.values, 'odds_ratio': np.exp(d3_model.params.values), 'p_value': d3_model.pvalues.values, 'ci_lower_or': np.exp(d3_model.conf_int()[0].values), 'ci_upper_or': np.exp(d3_model.conf_int()[1].values)})
d3_result.to_csv(REPORT_DIR / 'regression_d3_logistic.csv', index=False)
plt.figure(figsize=(9, max(5, len(d3_result)*.25))); coef = d3_result[d3_result.term != 'Intercept'].sort_values('odds_ratio'); plt.errorbar(coef.odds_ratio, range(len(coef)), xerr=[coef.odds_ratio-coef.ci_lower_or, coef.ci_upper_or-coef.odds_ratio], fmt='o'); plt.yticks(range(len(coef)), coef.term); plt.axvline(1,color='black',lw=1); plt.xscale('log'); plt.title('D3: odds ratio coefficient plot'); plt.xlabel('Odds ratio (95% CI)'); savefig('D3_logistic_coefficients.png')

                  Generalized Linear Model Regression Results                  
Dep. Variable:     has_recorded_change   No. Observations:                21498
Model:                             GLM   Df Residuals:                    21487
Model Family:                 Binomial   Df Model:                           10
Link Function:                   Logit   Scale:                          1.0000
Method:                           IRLS   Log-Likelihood:                -13455.
Date:                 Thu, 12 Nov 2026   Deviance:                       26911.
Time:                         10:02:11   Pearson chi2:                 2.15e+04
No. Iterations:                      5   Pseudo R-squ. (CS):            0.04086
Covariance Type:               cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Interc

## D4 — R², p-value và VIF

In [4]:
model_summary = pd.DataFrame([
    {'model':'D2 OLS multiple', 'n':int(d2_model.nobs), 'r_squared':d2_model.rsquared, 'adj_r_squared':d2_model.rsquared_adj, 'aic':d2_model.aic},
    {'model':'D3 GLM logistic', 'n':int(d3_model.nobs), 'r_squared':np.nan, 'adj_r_squared':np.nan, 'aic':d3_model.aic}
])
display(model_summary); model_summary.to_csv(REPORT_DIR / 'regression_model_summary.csv', index=False)
_, X = dmatrices('days_to_modification ~ code_lines + repo_commit_count + C(language) + C(ai_tool)', data=d2, return_type='dataframe')
vif = pd.DataFrame({'term': X.columns, 'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
display(vif); vif.to_csv(REPORT_DIR / 'regression_d2_vif.csv', index=False)

,model,n,r_squared,adj_r_squared,aic
0,D2 OLS multiple,5758,0.025237,0.023711,65397.363048
1,D3 GLM logistic,21498,NaN,NaN,26932.625760


,term,VIF
0,Intercept,1.000000
1,C(language)[T.python],1.081577
2,C(ai_tool)[T.claude],2.549730
3,C(ai_tool)[T.gemini_bard],1.120411
4,C(ai_tool)[T.github_copilot],1.247797
5,C(ai_tool)[T.llama_meta],1.040899
6,C(ai_tool)[T.other_ai],1.508705
7,C(ai_tool)[T.other_tool],1.003551
8,code_lines,1.039142
9,repo_commit_count,2.138556


## D5 — Residual plot
Residual plot kiểm tra xem sai số của D2 có hình dạng bất thường hay không.

In [5]:
pred = d2_model.get_prediction(d2).summary_frame()
plt.figure(figsize=(8,5)); sns.scatterplot(x=pred['mean'], y=d2_model.resid, s=12, alpha=.25); plt.axhline(0,color='red',ls='--'); plt.xlabel('Giá trị dự đoán'); plt.ylabel('Residual'); plt.title('D2 residual plot'); savefig('D5_residual_plot.png')
print('Các file kết quả đã lưu trong reports/ và reports/figures/.')

Các file kết quả đã lưu trong reports/ và reports/figures/.
